# Submission 2: Reflection and the Lightweight Challenge: SOLUTION KEY
### ME 323 Module 1 (staff only)

<img src="https://raw.githubusercontent.com/andrewvoss8-boop/core-me-data-science-activities-public/main/me323/Module1_drafts/figures/I_beam_dimensions.jpg" alt="I-beam dimensions" width="220">

Your beam has been printed and tested. Three jobs here:

1. Recall the module's ideas from memory.
2. Reflect on your measured result against your recorded prediction.
3. Design the lightest beam that confidently clears 700 N.

## 0. Recall

Write before computing. Corrections earn credit; unsupported bluffing does not.

1. Name the three modeled capacity branches and explain how the dominant-mode
   proxy is assigned. Which region of the (b, H_web) box does each own?
2. Pre-lab 1 calibrated σ_y, k, and c_s. For each: was it a correction or a
   confession? (One sentence each.)
3. A GP returns μ and σ at every design. Which one drove explore-vs-exploit,
   and what does ψ trade off?
4. The equation beam measured below its prediction; the GP beam measured above
   its central prediction. Give one plausible reason for each miss.
5. Name the four modeling lanes from Submission 1 and the one-line idea of each.

## 1. Your beam's test result

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

# Fixed geometry (you choose b and H_web; everything else is set)
B, TH, L = 10.0, 18.0, 150.0     # flange width, total height, test span (mm)
LP = 172.0                        # printed length (mm); overhangs the 150 mm span
KMASS = 0.2045                    # g/mm^2: mass per unit cross-section area at 172 mm

# Handbook starting values — Pre-lab 1 calibrates the three marked ones
SY = 76e6                         # Pa, PLA strength                 (calibrated)
K_LTB = 0.33                      # fixture effective-length factor  (calibrated)
CS = 1.0                          # web shear-strength multiplier    (calibrated)
E, G = 2.5e9, 2.5e9 / 2.6         # Young's / shear modulus (Pa) — fixed
C1, C2 = 1.35, 0.55               # LTB moment-gradient / load-height factors
URL = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
       "core-me-data-science-activities-public/main/data/student_beams_B10_L150.csv")
try:
    df = pd.read_csv(URL); print("loaded from GitHub")
except Exception:
    df = pd.read_csv("student_beams_B10_L150.csv"); print("loaded local copy")
df = df.rename(columns={"b_mm": "b", "H_web_mm": "H"})

def mass_g(b, H):
    A = b * H + B * (TH - H)      # cross-section area, mm^2
    return KMASS * A              # grams
df["mass_g"] = mass_g(df.b, df.H)
print(len(df), "tested beams")

new = pd.DataFrame([dict(beam_id=15, b=1.25, H=13.4, strength_N=483.0),
                    dict(beam_id=16, b=1.44, H=13.39, strength_N=509.7)])
new["mass_g"] = mass_g(new.b, new.H)
df = pd.concat([df, new], ignore_index=True)
df["str_to_weight"] = df.strength_N / df.mass_g

# >>> ENTER your group's final design and its measured result:
b_mine, H_mine = None, None        # your Submission 1 design (mm)
P_mine = None                      # measured failure load (N)
note_mine = ""                     # what the failure looked like
pred_median_sw = None              # copy model central prediction from Submission 1
pred_sigma_log = None              # copy epistemic sigma_log from Submission 1
if None not in (b_mine, H_mine, P_mine, pred_median_sw, pred_sigma_log):
    sw_mine = P_mine / mass_g(b_mine, H_mine)
    pred_lo_sw = pred_median_sw*np.exp(-2*pred_sigma_log)
    pred_hi_sw = pred_median_sw*np.exp(2*pred_sigma_log)
    inside_2sigma = pred_lo_sw <= sw_mine <= pred_hi_sw
    print(f"your beam: ({b_mine}, {H_mine}), {P_mine} N -> {sw_mine:.1f} N/g")
    print(f"recorded model interval: [{pred_lo_sw:.1f}, {pred_hi_sw:.1f}] N/g")
    print("inside recorded model +/-2 sigma interval:", inside_2sigma)
    print(f"class scoreboard: best tested so far {df.str_to_weight.max():.1f} N/g")

loaded from GitHub
14 tested beams


Reflect in the memo using the interval printed above. If the result lies
outside it, distinguish model-form error, print-to-print variability, and an
unmodeled failure mechanism. Do not call one test enough to identify which.

## 2. The lightweight challenge: hold 700 N, weigh as little as possible

Same 16 beams, same tools — different objective. Now strength is a
**constraint**, not the prize. The class-default confidence rule: require the
model's *pessimistic* strength (mean minus 2σ, in log space) to clear the
target:

$$P_{lo}(b, H) = e^{\,\mu_{\ln P}(b,H) - 2\sigma(b,H)} \;\ge\; 700\text{ N}$$

Among designs that pass, take the lightest. **FILL IN** the two marked lines.
(You may argue a different z than 2 in your memo — that is a risk posture,
not a math fact.)

In [2]:
def section_props(b, H):
    """I-section properties. b, H in mm; everything returned in METERS/SI."""
    tf = (TH - H) / 2.0
    b_, h_, B_, tf_ = b/1e3, H/1e3, B/1e3, tf/1e3
    c = (TH/1e3) / 2
    Ix = (b_*h_**3)/12 + 2*((B_*tf_**3)/12 + B_*tf_*(h_/2 + tf_/2)**2)
    Iy = (h_*b_**3)/12 + 2*(tf_*B_**3)/12
    def J_rect(x, y):
        short, long = min(x, y), max(x, y)
        r = short/long
        beta = 1 - 0.63*r + 0.052*r**5
        return (1/3)*beta*long*short**3
    J = J_rect(b_, h_) + 2*J_rect(tf_, B_)
    Cw = Iy*(h_ + tf_)**2/4
    return dict(Ix=Ix, Iy=Iy, J=J, Cw=Cw, c=c,
                b=b_, h=h_, tf=tf_, B=B_)

def P_bend(p, sy):
    return 4*sy*p["Ix"] / (p["c"] * L/1e3)
def P_shear(p, sy, cs):
    return 2 * (cs*sy/np.sqrt(3)) * p["b"]*p["h"]
def P_interaction_surrogate(Pb, Ps):
    return 1.0/np.sqrt(1/Pb**2 + 1/Ps**2)
def P_pointwise_yield(p, sy, n=801, return_detail=False):
    """Elastic first yield from co-located My/I and VQ/(It) stresses."""
    c, h2 = p["c"], p["h"]/2
    eps = max(c, 1.0)*1e-10
    y = np.unique(np.r_[np.linspace(0, c, n),
                        max(0, h2-eps), min(c, h2+eps)])
    in_web = y <= h2
    width = np.where(in_web, p["b"], p["B"])
    q_flange = p["B"]*p["tf"]*(h2 + p["tf"]/2)
    Q = np.where(
        in_web,
        q_flange + p["b"]*(h2-y)*(y+h2)/2,
        p["B"]*(c-y)*(y+c)/2,
    )
    sigma_per_N = (L/1e3)*y/(4*p["Ix"])
    tau_per_N = Q/(2*p["Ix"]*width)
    vm_per_N = np.sqrt(sigma_per_N**2 + 3*tau_per_N**2)
    loads = np.divide(sy, vm_per_N, out=np.full_like(vm_per_N, np.inf),
                      where=vm_per_N > 0)
    i = int(np.argmin(loads))
    if return_detail:
        return float(loads[i]), float(y[i]), float(sigma_per_N[i]), float(tau_per_N[i])
    return float(loads[i])
def P_LTB(p, sy, k):
    My = sy*p["Ix"]/p["c"]
    Lb, zg = k*L/1e3, p["c"]
    R = p["Cw"]/p["Iy"] + (Lb**2*G*p["J"])/(np.pi**2*E*p["Iy"]) + (C2*zg)**2
    Mcr = C1*np.pi**2*E*p["Iy"]/Lb**2 * (np.sqrt(R) - C2*zg)
    return 4*min(My, Mcr)/(L/1e3)
def capacity(b, H, sy, k, cs):
    p = section_props(b, H)
    return min(P_interaction_surrogate(P_bend(p, sy), P_shear(p, sy, cs)),
               P_LTB(p, sy, k))
def gov_mode(b, H, sy, k, cs):
    """Dominant pure-mode proxy, not an observed failure-mechanism label."""
    p = section_props(b, H)
    Pb, Ps, Pl = P_bend(p, sy), P_shear(p, sy, cs), P_LTB(p, sy, k)
    return "interaction/shear proxy" if Ps < min(Pb, Pl) else (
        "LTB" if Pl < 0.999*Pb else "bend")

SY_CAL, K_CAL, CS_CAL = 6.650e+07, 0.377, 2.25
P_TARGET = 700.0

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF

# class-default model: lane A (log str/w), 3% noise — swap in your own lane if you prefer
X = df[["b", "H"]].values
fmu, fsd = X.mean(0), X.std(0) + 1e-12
y = np.log(df.str_to_weight.values)
ymean = y.mean()
gp = GaussianProcessRegressor(C(1.0, (1e-3, 1e3))*RBF([1.0, 1.0], (1e-1, 30.0)),
                              alpha=0.03**2, normalize_y=False,
                              n_restarts_optimizer=5, random_state=0).fit((X-fmu)/fsd, y-ymean)

bg = np.linspace(1.25, 7.0, 60); Hg = np.linspace(5.0, 16.0, 60)
BB, HH = np.meshgrid(bg, Hg)
Xg = np.column_stack([BB.ravel(), HH.ravel()])
mu_c, std = gp.predict((Xg - fmu)/fsd, return_std=True)
mass_grid = mass_g(Xg[:, 0], Xg[:, 1])
# strength = str/w * mass, so in logs: ln P = (mu + ymean) + ln(mass)
mu_lnP = mu_c + ymean + np.log(mass_grid)

P_lo = np.exp(mu_lnP - 2*std)
feasible = P_lo >= P_TARGET
mean_strength = np.exp(mu_lnP)
masked = np.where(feasible, mass_grid, np.inf)
i = int(np.argmin(masked))
b_lt, H_lt = float(Xg[i, 0]), float(Xg[i, 1])
mean_feasible = mean_strength >= P_TARGET
i_mean = int(np.argmin(np.where(mean_feasible, mass_grid, np.inf)))
lighter_infeasible = (~feasible) & (mass_grid < mass_grid[i])
if lighter_infeasible.any():
    j = int(np.argmax(np.where(lighter_infeasible, mass_grid, -np.inf)))
else:
    j = None
print(f"LIGHTWEIGHT DESIGN: b = {b_lt:.2f} mm, H_web = {H_lt:.2f} mm")
print(f"  mass {mass_grid[i]:.1f} g,  P_lo {P_lo[i]:.0f} N,  "
      f"model mean {mean_strength[i]:.0f} N")
print(f"  uncertainty allowance: model mean - P_lo = "
      f"{mean_strength[i]-P_lo[i]:.0f} N")
print(f"  mean-only lightest design: b={Xg[i_mean,0]:.2f}, H={Xg[i_mean,1]:.2f}, "
      f"mass={mass_grid[i_mean]:.1f} g, mean={mean_strength[i_mean]:.0f} N, "
      f"P_lo={P_lo[i_mean]:.0f} N")
print(f"  mass added by the 2-sigma rule versus mean-only: "
      f"{mass_grid[i]-mass_grid[i_mean]:.1f} g")
if j is not None:
    print(f"  nearest lighter infeasible grid point: b={Xg[j,0]:.2f}, "
          f"H={Xg[j,1]:.2f}, mass={mass_grid[j]:.3f} g "
          f"({mass_grid[i]-mass_grid[j]:.3f} g lighter), P_lo={P_lo[j]:.0f} N")
print(f"  calibrated-physics check: {capacity(b_lt, H_lt, SY_CAL, K_CAL, CS_CAL):.0f} N, "
      f"mode {gov_mode(b_lt, H_lt, SY_CAL, K_CAL, CS_CAL)}")
print("\nCHECKPOINT (class-default model): you should arrive at "
      "b = 4.27, H_web = 13.20, mass = 21.3 g.")
print("If you are not getting that, check your work or talk to a TA.")

LIGHTWEIGHT DESIGN: b = 4.27 mm, H_web = 13.20 mm
  mass 21.3 g,  P_lo 703 N,  model mean 739 N
  uncertainty allowance: model mean - P_lo = 36 N
  mean-only lightest design: b=4.37, H=15.07, mass=19.5 g, mean=701 N, P_lo=651 N
  mass added by the 2-sigma rule versus mean-only: 1.9 g
  nearest lighter infeasible grid point: b=3.10, H=10.97, mass=21.340 g (0.002 g lighter), P_lo=662 N
  calibrated-physics check: 739 N, mode bend

CHECKPOINT (class-default model): you should arrive at b = 4.27, H_web = 13.20, mass = 21.3 g.
If you are not getting that, check your work or talk to a TA.


## Memo

1. Margin: use the printed mean, lower bound, mean-only design, robust design,
   and nearest lighter infeasible candidate. State the comparison in newtons and grams.
2. `z`: defend 2 or price another value. A one-sided standard Gaussian tail
   beyond `z=2` is about 2.3%, but only if this posterior uncertainty is calibrated.
3. Physics veto: cite the calibrated capacity and dominant-mode proxy. If it
   disagrees with the GP constraint, explain which evidence you prioritize.
4. One more test: provide coordinates and say whether mean, epistemic sigma,
   or proximity to the feasibility boundary motivates them.

## KEY: memo targets

1. The class-default output directly supplies the required quantities and a
   defined neighboring grid point. Students should not compare against the
   globally lightest infeasible beam.
2. `z=2` corresponds to a 2.28% one-sided tail only for a calibrated Gaussian
   predictive distribution. Fifteen to sixteen sparse points, fitted kernel
   choices, and model-form error make that probability provisional.
3. Compare the printed GP lower bound with the empirical-physics capacity and
   proxy mode. Agreement is supporting evidence, not independent validation,
   because both were informed by the same small campaign.
4. A useful extra test lies near the active lower-boundary contour, especially
   where epistemic uncertainty or an observed separation mechanism could change
   feasibility.